# Tutorial 04: SO-ARM101 Robot

[![ Click here to deploy.](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-35QaaoiXx6VDmVNBOEtmpLs9VBs)

This tutorial demonstrates how to load and control the **SO-ARM101** robot arm in Newton using URDF import. The SO-ARM101 is a 6-DOF robot arm designed by The Robot Studio, commonly used for robotics education and research.

## Learning Objectives

By the end of this tutorial, you will:

1. **Load a URDF robot** — Import SO-ARM101 from URDF with mesh assets
2. **Explore robot kinematics** — Understand joint structure and limits
3. **Control joint positions** — Set target joint angles
4. **Perform inverse kinematics** — Move the gripper to target positions
5. **Visualize the robot** — Use Newton's Rerun viewer

---


## Setup and Imports


In [1]:
import newton
import newton.ik as ik
import warp as wp
import numpy as np
from tqdm.notebook import trange
import os

# Initialize Warp
wp.init()

# Set NumPy print options for cleaner output
np.set_printoptions(precision=4, suppress=True, linewidth=100)

print("="*60)
print("🤖 SO-ARM101 Newton Physics Simulation")
print("="*60)


Warp 1.11.0.dev20251123 initialized:
   Git commit: 8b8f0b85ca54c0026574f834764e26615056aef6
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L40S" (44 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0.dev20251123
🤖 SO-ARM101 Newton Physics Simulation


---

## Part 1: About the SO-ARM101

The **SO-ARM101** is a 6-DOF robotic arm with the following specifications:

### Joint Structure

| Joint | Name | Type | Description | Limits |
|-------|------|------|-------------|--------|
| 1 | `shoulder_pan` | Revolute | Base rotation | ±110° |
| 2 | `shoulder_lift` | Revolute | Shoulder pitch | ±100° |
| 3 | `elbow_flex` | Revolute | Elbow pitch | ±97° |
| 4 | `wrist_flex` | Revolute | Wrist pitch | ±95° |
| 5 | `wrist_roll` | Revolute | Wrist roll | ±157°/163° |
| 6 | `gripper` | Revolute | Gripper open/close | -10°/+100° |

### Kinematic Chain

```
[World] → base_link → shoulder_link → upper_arm_link → lower_arm_link → wrist_link → gripper_link → gripper_frame_link
             ↓              ↓               ↓               ↓              ↓              ↓
        (shoulder_pan) (shoulder_lift) (elbow_flex)   (wrist_flex)   (wrist_roll)   (gripper)
```

The robot uses **STS3215** servo motors with 3D-printed structural components.


---

## Part 2: Load SO-ARM101 from URDF

Newton's `ModelBuilder.add_urdf()` function handles URDF parsing including mesh loading.


In [2]:
# Path to SO-ARM101 URDF
# Auto-clone SO-ARM100 if not present

import subprocess

# In Jupyter, the cwd is typically the notebook's directory
assets_dir = "assets"
so_arm_dir = os.path.join(assets_dir, "SO-ARM100")
urdf_path = os.path.join(so_arm_dir, "Simulation/SO101/so101_new_calib.urdf")

# Clone if not present
if not os.path.exists(urdf_path):
    print("📥 SO-ARM100 not found, cloning from GitHub...")
    os.makedirs(assets_dir, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/TheRobotStudio/SO-ARM100.git",
        so_arm_dir
    ], check=True)
    print("✅ Clone complete!")

print(f"✅ URDF: {urdf_path}")


✅ URDF: assets/SO-ARM100/Simulation/SO101/so101_new_calib.urdf


In [3]:
# Create ModelBuilder
builder = newton.ModelBuilder()

# Set default parameters for the robot
builder.default_body_armature = 0.01
builder.default_joint_cfg.armature = 0.01
builder.default_joint_cfg.target_ke = 1000.0  # Position gain
builder.default_joint_cfg.target_kd = 50.0    # Damping
builder.default_shape_cfg.ke = 1.0e4
builder.default_shape_cfg.kd = 1.0e2
builder.default_shape_cfg.kf = 1.0e2
builder.default_shape_cfg.mu = 1.0

# Parse the URDF file
# The robot is mounted on a surface, so we use floating=False for a fixed base
robot_height = 0.0  # Base height offset
builder.add_urdf(
    urdf_path,
    xform=wp.transform(wp.vec3(0.0, 0.0, robot_height), wp.quat_identity()),
    floating=False,  # Fixed base
    enable_self_collisions=False,
    ignore_inertial_definitions=False,  # Use URDF inertial parameters
)

# Add ground plane
builder.add_ground_plane()

print(f"\n✅ SO-ARM101 loaded successfully!")
print(f"   Bodies: {builder.body_count}")
print(f"   Joints: {builder.joint_count}")


Module newton._src.geometry.inertia 829810a load on device 'cuda:0' took 1.18 ms  (cached)

✅ SO-ARM101 loaded successfully!
   Bodies: 8
   Joints: 8


---

## Part 3: Finalize Model and Explore Structure


In [4]:
# Finalize the model
model = builder.finalize()

print("="*60)
print("SO-ARM101 Model Summary")
print("="*60)
print(f"Bodies: {model.body_count}")
print(f"Joints: {model.joint_count}")
print(f"DOF: {model.joint_dof_count}")
print(f"Joint coordinates: {model.joint_coord_count}")
print(f"Shapes: {model.shape_count}")

print("\n📋 Body Names:")
for i, key in enumerate(model.body_key):
    print(f"  [{i}] {key}")

print("\n🔗 Joint Names:")
for i, key in enumerate(model.joint_key):
    print(f"  [{i}] {key}")


Module validate_and_correct_inertia_kernel_4e499976 4e49997 load on device 'cuda:0' took 0.49 ms  (cached)
Module count_contact_points_cf171c27 cf171c2 load on device 'cuda:0' took 0.35 ms  (cached)
SO-ARM101 Model Summary
Bodies: 8
Joints: 8
DOF: 6
Joint coordinates: 6
Shapes: 35

📋 Body Names:
  [0] base_link
  [1] shoulder_link
  [2] upper_arm_link
  [3] lower_arm_link
  [4] wrist_link
  [5] gripper_link
  [6] gripper_frame_link
  [7] moving_jaw_so101_v1_link

🔗 Joint Names:
  [0] fixed_base
  [1] shoulder_pan
  [2] shoulder_lift
  [3] elbow_flex
  [4] wrist_flex
  [5] wrist_roll
  [6] gripper_frame_joint
  [7] gripper


In [5]:
# Display joint limits
print("\n📐 Joint Limits:")
print("-"*60)
print(f"{'Joint':<20} {'Lower (rad)':<15} {'Upper (rad)':<15} {'Range (deg)'}")
print("-"*60)

joint_limit_lower = model.joint_limit_lower.numpy()
joint_limit_upper = model.joint_limit_upper.numpy()

# Skip the first joint (fixed base) and iterate over actuated joints
actuated_joint_names = ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
for i, name in enumerate(actuated_joint_names):
    if i < len(joint_limit_lower):
        lower = joint_limit_lower[i]
        upper = joint_limit_upper[i]
        range_deg = np.degrees(upper - lower)
        print(f"{name:<20} {lower:<15.3f} {upper:<15.3f} {range_deg:.1f}°")



📐 Joint Limits:
------------------------------------------------------------
Joint                Lower (rad)     Upper (rad)     Range (deg)
------------------------------------------------------------
shoulder_pan         -1.920          1.920           220.0°
shoulder_lift        -1.745          1.745           200.0°
elbow_flex           -1.690          1.690           193.7°
wrist_flex           -1.658          1.658           190.0°
wrist_roll           -2.744          2.841           320.0°
gripper              -0.175          1.745           110.0°


---

## Part 4: Initial Visualization


In [6]:
# Create state and evaluate forward kinematics
state = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state)

# Create viewer
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)
viewer.set_model(model)
viewer.log_state(state)

print("✅ Initial robot state visualized")
viewer


Module newton._src.sim.articulation 46448df load on device 'cuda:0' took 2.88 ms  (cached)
Module newton._src.viewer.viewer_rerun ef719a2 load on device 'cuda:0' took 0.39 ms  (cached)
Module map_normalize e9a5d4c load on device 'cuda:0' took 0.40 ms  (cached)
Module newton._src.viewer.kernels 1205f75 load on device 'cuda:0' took 1.05 ms  (cached)
✅ Initial robot state visualized


HTML(value='<div id="9bde3af4-c446-4028-8288-cc8dabf5eac1"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…